In [1]:
# Incremental Merkle tree

ZERO = 0
DEPTH = 4

def hash(a, b):
    return f'h({a},{b})'

class MerkleTree:
    def __init__(self):
        # Calculate tree with all leaves = ZERO
        zeros = [ZERO] * DEPTH
        z = ZERO
        for i in range(DEPTH):
            zeros[i] = z
            # Next level
            z = hash(z, z)

        self.root = z
        # Zero hash at each level
        self.zeros = zeros
        # Right most left node at each level
        self.nodes = zeros[:]
        # Next leaf index
        self.next = 0
        # Leaves
        self.vals = [ZERO] * (2**DEPTH)

    def insert(self, val):
        assert self.next < 2 ** DEPTH

        i = self.next
        self.vals[i] = val
        self.next += 1
        h = val

        for level in range(DEPTH):
            left, right = (ZERO, ZERO)
            if i % 2 == 0:
                left = h
                right = self.zeros[level]
                self.nodes[level] = left
            else:
                left = self.nodes[level]
                right = h
            h = hash(left, right)
            i //= 2

        self.root = h

    def insert_many(self, vals):
        count = len(vals)

        if count == 0:
            return

        assert self.next + count <= 2 ** DEPTH

        # Starting index
        s = self.next
        for i in range(count):
            self.vals[s + i] = vals[i]
        self.next += count

        n = count
        hs = vals[:]

        for level in range(DEPTH):
            # Next level index to store hash(left, right) into hs[k]
            k = 0

            # Starting index is a right node
            # Pair this right node with hs[0],
            # calculate hash of 1 level above 
            # and store it into hs[0]
            i = 0
            if s % 2 == 1:
                hs[0] = hash(self.nodes[level], hs[0])
                i += 1
                k += 1

            left = None
            while i < n:
                left = hs[i]
                right = hs[i + 1] if i + 1 < n else self.zeros[level]
                hs[k] = hash(left, right)
                i += 2
                k += 1
            
            if left != None:
                self.nodes[level] = left

            n = k
            # Starting index of next level
            s //= 2

        self.root = hs[0]

In [2]:
import random

def test():
    tree = MerkleTree()
    ref = MerkleTree()

    i = 0
    while i < 2 ** DEPTH:
        r = random.randint(1, 2 ** DEPTH - i)
        vals = list(range(i + 1, i + r + 1))
        tree.insert_many(vals)
        for v in vals:
            ref.insert(v)
        i += r

    assert tree.root == ref.root, f"root mismatch: {tree.root} != {ref.root}"
    assert tree.nodes == ref.nodes, f"nodes mismatch: {tree.nodes} != {ref.nodes}"

for _ in range(1000):
    test()